# 📖 Notebook 2: Real-Time Aggregation with Sliding Windows

In Notebook 1 we ingested click events into Kafka and stored them in PostgreSQL. But querying raw events is **too slow** for advertisers who need instant dashboards.

The solution? **Pre-aggregate** clicks into 1-minute buckets *as they stream in*. When an advertiser asks "how many clicks did my Nike ad get in the last hour?", we just SUM 60 pre-computed rows instead of scanning millions of raw events.

```
Raw events (millions)          Pre-aggregated (one row per ad per minute)
┌────────────────────┐         ┌──────────────────────────────────┐
│ ad=1, 10:00:03     │         │ ad=1  | 10:00 | 47 clicks       │
│ ad=1, 10:00:15     │  ───▶   │ ad=1  | 10:01 | 52 clicks       │
│ ad=1, 10:00:47     │         │ ad=2  | 10:00 | 31 clicks       │
│ ad=2, 10:00:02     │         │ ...                              │
│ ...millions more   │         └──────────────────────────────────┘
└────────────────────┘         Much faster to query!
```

## Learning Objectives

By the end of this notebook you will understand:
- The difference between **event time** and **processing time**
- How **tumbling windows** (fixed 1-minute buckets) work
- How to build an aggregation consumer that writes to `click_aggregates`
- How **late-arriving events** can land in the wrong window and how to handle them
- Why pre-aggregation makes advertiser queries sub-second

## 🛠️ Setup

Make sure the infrastructure is running:

```bash
cd system-designs/ad-click-aggregator
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL): http://localhost:8080
- **Kafka UI**: http://localhost:8081

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import json
import time
import uuid
import random
from datetime import datetime, timezone, timedelta
from collections import defaultdict
from confluent_kafka import Producer, Consumer, KafkaError
from confluent_kafka.admin import AdminClient, NewTopic

# ── Connection settings ─────────────────────────────────────
DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "adclick_demo", "user": "demo", "password": "demo"
}
KAFKA_BROKER = "localhost:9094"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Verify connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
    admin.list_topics(timeout=5)
    print("✅ Kafka")
except Exception as e:
    print(f"❌ Kafka: {e}")

## ⏰ Event Time vs Processing Time

This is one of the most important concepts in stream processing. Let's understand the difference:

| Concept | Definition | Example |
|---------|------------|----------|
| **Event time** | When the click *actually happened* on the user's device | `10:00:03 UTC` |
| **Processing time** | When our server *received and processed* the event | `10:00:05 UTC` |

Why the difference? Network delays, server load, or the user's phone being briefly offline can all cause events to arrive late.

**We always aggregate by event time** because that's when the click truly occurred. Using processing time would produce inaccurate counts.

In [ ]:
# Let's demonstrate the problem with a concrete example

print("⏰ Event Time vs Processing Time")
print("=" * 60)
print()

# Simulate 5 clicks that happened at known event times
# but arrived at our server in a different order
clicks = [
    {"event_time": "10:00:03", "processed_at": "10:00:04", "note": "arrived on time"},
    {"event_time": "10:00:15", "processed_at": "10:00:16", "note": "arrived on time"},
    {"event_time": "10:00:45", "processed_at": "10:00:46", "note": "arrived on time"},
    {"event_time": "10:00:58", "processed_at": "10:01:03", "note": "⚠️ arrived LATE (5s delay)"},
    {"event_time": "10:01:02", "processed_at": "10:01:02", "note": "arrived on time"},
]

print(f"{'Event Time':<14} {'Processed At':<14} {'Note'}")
print("-" * 60)
for c in clicks:
    print(f"{c['event_time']:<14} {c['processed_at']:<14} {c['note']}")

print()
print("📊 If we aggregate by EVENT TIME (correct):")
print("   Window 10:00 → 4 clicks  (events at :03, :15, :45, :58)")
print("   Window 10:01 → 1 click   (event at :02)")
print()
print("📊 If we aggregate by PROCESSING TIME (wrong):")
print("   Window 10:00 → 3 clicks  (processed at :04, :16, :46)")
print("   Window 10:01 → 2 clicks  (processed at :02, :03)")
print()
print("⚠️  The late event at 10:00:58 would be miscounted in the 10:01 window!")
print("   Always use EVENT TIME for accurate aggregation.")

## 🪟 Tumbling Windows

A **tumbling window** is a fixed-size, non-overlapping time bucket. For our system, each window is exactly 1 minute long:

```
Time ──────────────────────────────────────────────▶

│  Window 1   │  Window 2   │  Window 3   │
│ 10:00-10:01 │ 10:01-10:02 │ 10:02-10:03 │
│  ● ● ●  ●  │  ●  ●       │  ● ● ● ● ● │
│  4 clicks   │  2 clicks   │  5 clicks   │
```

Every click goes into exactly **one** window based on its event time. No overlaps, no gaps.

Let's build a function that calculates which window a click belongs to.

In [ ]:
def get_window_start(event_time_str: str, window_seconds: int = 60) -> datetime:
    """
    Given an event time, return the start of its tumbling window.

    Example: event at 10:00:43 with 60s windows → window starts at 10:00:00
    """
    event_time = datetime.fromisoformat(event_time_str)
    # Truncate to the start of the window
    timestamp = event_time.timestamp()
    window_start_ts = (timestamp // window_seconds) * window_seconds
    return datetime.fromtimestamp(window_start_ts, tz=timezone.utc)

# Demonstrate with some example times
examples = [
    "2026-03-31T10:00:03+00:00",
    "2026-03-31T10:00:45+00:00",
    "2026-03-31T10:00:59+00:00",
    "2026-03-31T10:01:00+00:00",
    "2026-03-31T10:01:30+00:00",
]

print("🪟 Tumbling Window Assignment (1-minute windows)")
print("=" * 55)
print(f"{'Event Time':<30} {'Window Start':<20}")
print("-" * 55)
for et in examples:
    ws = get_window_start(et)
    short_event = et[11:19]
    short_window = ws.strftime("%H:%M:%S")
    print(f"  {short_event:<28} {short_window:<20}")

print()
print("💡 10:00:59 and 10:01:00 are just 1 second apart")
print("   but land in DIFFERENT windows. That's tumbling windows!")

## 🚀 Step 1 — Produce Click Events with Controlled Timestamps

Let's produce events with specific timestamps so we can verify our aggregation is correct.

In [ ]:
topic_name = "ad-clicks-agg"

# Create topic if needed
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
existing = admin.list_topics(timeout=10).topics
if topic_name not in existing:
    futures = admin.create_topics([NewTopic(topic_name, num_partitions=3, replication_factor=1)])
    for t, f in futures.items():
        f.result()
    print(f"✅ Created topic '{topic_name}'")
else:
    print(f"ℹ️  Topic '{topic_name}' already exists")

# Produce events across 3 one-minute windows
producer = Producer({"bootstrap.servers": KAFKA_BROKER})

base_time = datetime(2026, 3, 31, 10, 0, 0, tzinfo=timezone.utc)

# We'll create a known distribution so we can verify our aggregation
# Window 10:00 → ad 1 gets 5 clicks, ad 2 gets 3 clicks
# Window 10:01 → ad 1 gets 2 clicks, ad 2 gets 7 clicks
# Window 10:02 → ad 1 gets 4 clicks, ad 2 gets 1 click
planned_events = []
distributions = [
    (0, 1, 5),   # minute 0, ad 1, 5 clicks
    (0, 2, 3),   # minute 0, ad 2, 3 clicks
    (1, 1, 2),   # minute 1, ad 1, 2 clicks
    (1, 2, 7),   # minute 1, ad 2, 7 clicks
    (2, 1, 4),   # minute 2, ad 1, 4 clicks
    (2, 2, 1),   # minute 2, ad 2, 1 click
]

for minute_offset, ad_id, count in distributions:
    for i in range(count):
        event_time = base_time + timedelta(minutes=minute_offset, seconds=random.randint(0, 59))
        event = {
            "ad_id": ad_id,
            "impression_id": str(uuid.uuid4()),
            "user_id": f"user_{random.randint(1, 200)}",
            "ip_address": f"10.0.{random.randint(1,254)}.{random.randint(1,254)}",
            "user_agent": "Mozilla/5.0",
            "event_time": event_time.isoformat()
        }
        planned_events.append(event)

# Shuffle to simulate out-of-order arrival (realistic!)
random.shuffle(planned_events)

for event in planned_events:
    producer.produce(
        topic=topic_name,
        key=str(event["ad_id"]),
        value=json.dumps(event)
    )

producer.flush(timeout=10)

print(f"✅ Produced {len(planned_events)} events (shuffled to simulate out-of-order)")
print()
print("📊 Expected aggregation:")
print(f"   Window 10:00 → ad 1: 5 clicks, ad 2: 3 clicks")
print(f"   Window 10:01 → ad 1: 2 clicks, ad 2: 7 clicks")
print(f"   Window 10:02 → ad 1: 4 clicks, ad 2: 1 click")

## 🔄 Step 2 — Build the Aggregation Consumer

This is the heart of the system. Our consumer will:

1. **Read** click events from Kafka
2. **Assign** each event to a 1-minute window using its **event time**
3. **Accumulate** counts in memory (a dictionary)
4. **Flush** the counts to the `click_aggregates` table

In production, Flink does this automatically with watermarks and exactly-once guarantees. Here, we'll build a simplified version to understand the mechanics.

In [ ]:
def aggregate_clicks(topic: str, max_events: int = 100, timeout_s: int = 15):
    """
    Consume events from Kafka and aggregate into 1-minute windows.

    Returns a dict: {(ad_id, window_start): {"click_count": N, "unique_users": set()}}
    """
    consumer = Consumer({
        "bootstrap.servers": KAFKA_BROKER,
        "group.id": "agg-consumer-" + str(uuid.uuid4())[:8],
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False
    })
    consumer.subscribe([topic])

    # In-memory aggregation state
    # Key: (ad_id, window_start_str) → Value: {click_count, unique_users}
    windows = defaultdict(lambda: {"click_count": 0, "unique_users": set()})

    consumed = 0
    start = time.time()

    while time.time() - start < timeout_s and consumed < max_events:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            break

        event = json.loads(msg.value().decode("utf-8"))
        ad_id = event["ad_id"]
        event_time = event["event_time"]
        user_id = event.get("user_id", "anonymous")

        # Assign to window using EVENT TIME (not processing time!)
        window_start = get_window_start(event_time)
        key = (ad_id, window_start.isoformat())

        windows[key]["click_count"] += 1
        windows[key]["unique_users"].add(user_id)

        consumed += 1

    consumer.close()
    return consumed, dict(windows)

# Run the aggregation
consumed, windows = aggregate_clicks(topic_name, max_events=50)

print(f"📥 Consumed {consumed} events")
print(f"📊 Aggregated into {len(windows)} windows:\n")

print(f"{'Ad ID':<8} {'Window Start':<22} {'Clicks':>8} {'Unique Users':>13}")
print("-" * 55)
for (ad_id, window_start), data in sorted(windows.items()):
    short_window = window_start[11:16]  # just HH:MM
    print(f"{ad_id:<8} {short_window:<22} {data['click_count']:>8} {len(data['unique_users']):>13}")

print("\n✅ Compare these with the expected counts above!")

## 💾 Step 3 — Flush Aggregates to PostgreSQL

Now let's write the aggregated data to the `click_aggregates` table. We use **UPSERT** (`ON CONFLICT ... DO UPDATE`) so that if more events arrive for the same window, we just add to the existing count.

This is exactly what Flink does when it flushes its window state to the database.

In [ ]:
def flush_to_postgres(windows: dict):
    """
    Write aggregated window data to the click_aggregates table.
    Uses UPSERT so repeated flushes accumulate correctly.
    """
    conn = get_db()
    cur = conn.cursor()

    upsert_sql = """
        INSERT INTO click_aggregates (ad_id, window_start, click_count, unique_users)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (ad_id, window_start)
        DO UPDATE SET
            click_count = click_aggregates.click_count + EXCLUDED.click_count,
            unique_users = click_aggregates.unique_users + EXCLUDED.unique_users,
            updated_at = CURRENT_TIMESTAMP
    """

    flushed = 0
    for (ad_id, window_start), data in windows.items():
        cur.execute(upsert_sql, (
            ad_id,
            window_start,
            data["click_count"],
            len(data["unique_users"])
        ))
        flushed += 1

    conn.commit()
    conn.close()
    return flushed

flushed = flush_to_postgres(windows)
print(f"✅ Flushed {flushed} window aggregates to PostgreSQL")

## 📊 Step 4 — Advertiser Query: Fast!

Now let's see how fast the advertiser query is when it reads from the pre-aggregated table instead of scanning millions of raw events.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Fast query: just SUM the pre-aggregated rows
start = time.time()
cur.execute("""
    SELECT a.title, ca.window_start, ca.click_count, ca.unique_users
    FROM click_aggregates ca
    JOIN ads a ON a.id = ca.ad_id
    ORDER BY ca.window_start, a.title
""")
rows = cur.fetchall()
elapsed = (time.time() - start) * 1000
conn.close()

print(f"⚡ Query completed in {elapsed:.1f} ms\n")
print(f"{'Ad Title':<42} {'Window':<22} {'Clicks':>7} {'Unique':>7}")
print("-" * 82)
for row in rows:
    window_str = row[1].strftime("%Y-%m-%d %H:%M") if hasattr(row[1], 'strftime') else str(row[1])[:16]
    print(f"{row[0]:<42} {window_str:<22} {row[2]:>7} {row[3]:>7}")

print()
print("💡 This query touches only a few rows, not millions.")
print("   An advertiser asking 'how many clicks in the last hour?'")
print("   scans at most 60 rows per ad (one per minute) — instant!")

## ⚠️ Step 5 — Late-Arriving Events

What happens when a click event arrives *after* we've already closed and flushed its window?

For example, a user clicks an ad at 10:00:58 but due to network delays, the event doesn't reach our consumer until 10:01:05 — after the 10:00 window has been flushed.

### Strategies for Late Events

| Strategy | How It Works | Trade-off |
|----------|-------------|----------|
| **Drop** | Ignore events that arrive after the window closes | Simple but loses data |
| **Update** | Reopen the window and update the aggregate | Accurate but more complex |
| **Watermark** | Wait a grace period before closing the window | Good balance |

In our system, we use the **Update** strategy via UPSERT — late events simply add to the existing aggregate. Let's see it in action.

In [ ]:
# Simulate a late-arriving event
print("⚠️  Simulating a Late-Arriving Event")
print("=" * 50)

# Check current aggregate for ad 1 at 10:00
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT click_count FROM click_aggregates
    WHERE ad_id = 1 AND window_start = '2026-03-31 10:00:00+00'
""")
row = cur.fetchone()
before_count = row[0] if row else 0
print(f"\n📊 BEFORE: ad_id=1, window 10:00 has {before_count} clicks")

# A late event arrives for the 10:00 window
late_event_window = {
    (1, "2026-03-31T10:00:00+00:00"): {
        "click_count": 1,
        "unique_users": {"user_late_999"}
    }
}

print("\n⏰ Late event arrives: ad_id=1, event_time=10:00:58, arrived at 10:01:05")
print("   Our UPSERT adds it to the existing window...")

flush_to_postgres(late_event_window)

# Check after
cur.execute("""
    SELECT click_count FROM click_aggregates
    WHERE ad_id = 1 AND window_start = '2026-03-31 10:00:00+00'
""")
row = cur.fetchone()
after_count = row[0] if row else 0
conn.close()

print(f"\n📊 AFTER: ad_id=1, window 10:00 has {after_count} clicks")
print(f"   (+{after_count - before_count} from the late event)")
print("\n✅ The UPSERT handled the late event correctly!")
print("   No data was lost — the aggregate was updated in place.")

## 🔄 Step 6 — Full Pipeline: Produce → Aggregate → Query

Let's run the complete pipeline end-to-end: produce a burst of events, aggregate them, flush, and query.

In [ ]:
# Clean slate for this demo
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
conn.commit()
conn.close()

# ── PRODUCE ──
pipeline_topic = "ad-clicks-pipeline"
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
existing = admin.list_topics(timeout=10).topics
if pipeline_topic not in existing:
    futures = admin.create_topics([NewTopic(pipeline_topic, num_partitions=3, replication_factor=1)])
    for t, f in futures.items():
        f.result()

producer = Producer({"bootstrap.servers": KAFKA_BROKER})

base = datetime(2026, 3, 31, 14, 0, 0, tzinfo=timezone.utc)
num_events = 200

for _ in range(num_events):
    ad_id = random.randint(1, 5)  # focus on 5 ads
    offset_seconds = random.randint(0, 299)  # spread across 5 minutes
    event_time = base + timedelta(seconds=offset_seconds)
    event = {
        "ad_id": ad_id,
        "impression_id": str(uuid.uuid4()),
        "user_id": f"user_{random.randint(1, 100)}",
        "ip_address": f"10.0.1.{random.randint(1,254)}",
        "user_agent": "Mozilla/5.0",
        "event_time": event_time.isoformat()
    }
    producer.produce(pipeline_topic, key=str(ad_id), value=json.dumps(event))

producer.flush(timeout=10)
print(f"📤 Produced {num_events} events across 5 minutes")

# ── AGGREGATE ──
consumed, windows = aggregate_clicks(pipeline_topic, max_events=num_events)
print(f"📥 Consumed {consumed} events, aggregated into {len(windows)} windows")

# ── FLUSH ──
flushed = flush_to_postgres(windows)
print(f"💾 Flushed {flushed} aggregates to PostgreSQL")

# ── QUERY ──
conn = get_db()
cur = conn.cursor()

# Advertiser dashboard: total clicks per ad across all windows
start = time.time()
cur.execute("""
    SELECT a.title, SUM(ca.click_count) AS total_clicks,
           SUM(ca.unique_users) AS est_unique
    FROM click_aggregates ca
    JOIN ads a ON a.id = ca.ad_id
    GROUP BY a.id, a.title
    ORDER BY total_clicks DESC
""")
rows = cur.fetchall()
elapsed = (time.time() - start) * 1000
conn.close()

print(f"\n⚡ Advertiser dashboard query: {elapsed:.1f} ms\n")
print(f"{'Ad Title':<42} {'Total Clicks':>13} {'~Unique Users':>14}")
print("-" * 72)
for row in rows:
    print(f"{row[0]:<42} {row[1]:>13} {row[2]:>14}")

print("\n✅ Full pipeline: Produce → Aggregate → Query — all in seconds!")

## 🌊 Step 7 — Sliding Windows

Tumbling windows are great for "how many clicks in minute 10:00?" But advertisers often want **rolling metrics** like "how many clicks in the *last 5 minutes*, updated every minute?" That's a **sliding window**.

```
Time ──────────────────────────────────────────────▶

Tumbling (1-min, non-overlapping):
│ 10:00 │ 10:01 │ 10:02 │ 10:03 │ 10:04 │

Sliding (5-min window, 1-min step — overlapping!):
│───── 10:00-10:05 ─────│
        │───── 10:01-10:06 ─────│
                │───── 10:02-10:07 ─────│
```

A click at 10:02 contributes to **five** sliding windows (10:02–10:07, 10:01–10:06, 10:00–10:05, …). That sounds expensive — but there's a beautiful trick.

### The Trick: Sum Tumbling Windows

We already have 1-minute tumbling aggregates in `click_aggregates`. A 5-minute sliding total is just **the sum of the last 5 tumbling rows**. We don't need to re-process raw events — we compose sliding answers from the pre-aggregated data.

This is exactly what real systems (Flink, Druid, Pinot) do under the hood.


In [ ]:
# Compute 5-minute sliding totals by summing 1-minute tumbling rows.
# We reuse the aggregates produced in Step 2/3 above.

conn = get_db()
cur = conn.cursor()

# A SQL window function SUMs the previous 4 rows + current row per ad,
# giving us a rolling 5-minute total that slides every minute.
cur.execute("""
    SELECT
        a.title,
        ca.window_start,
        ca.click_count AS clicks_in_minute,
        SUM(ca.click_count) OVER (
            PARTITION BY ca.ad_id
            ORDER BY ca.window_start
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS rolling_5min_total
    FROM click_aggregates ca
    JOIN ads a ON a.id = ca.ad_id
    ORDER BY a.title, ca.window_start
""")
rows = cur.fetchall()
conn.close()

print(f"🌊 5-Minute Sliding Window (step = 1 minute)")
print("=" * 78)
print(f"{'Ad':<38} {'Window Start':<20} {'This Min':>8} {'Rolling 5m':>12}")
print("-" * 78)
for title, window_start, minute_clicks, rolling in rows:
    print(f"{title:<38} {window_start.strftime('%Y-%m-%d %H:%M'):<20} "
          f"{minute_clicks:>8} {rolling:>12}")

print()
print("💡 Each 'Rolling 5m' value is the SUM of the last 5 tumbling rows.")
print("   No raw-event scan needed — the pre-aggregates already hold the answer.")
print("   This is why pre-aggregation is the foundation of every streaming metric.")


### Sliding Windows in Pure Python (No SQL)

Want to see it without SQL? Here's the same logic in pure Python — useful if you're computing sliding totals in a stream processor like Flink or Kafka Streams.


In [ ]:
from collections import deque

def sliding_totals(per_minute_counts, window_minutes: int = 5):
    """Given a list of per-minute click counts, return the rolling window_minutes
    total for each minute. Uses a deque for O(1) amortized updates."""
    window = deque()
    running_sum = 0
    result = []
    for count in per_minute_counts:
        window.append(count)
        running_sum += count
        if len(window) > window_minutes:
            running_sum -= window.popleft()
        result.append(running_sum)
    return result

# Pretend ad 1 had these click counts per minute for 10 minutes
per_minute = [5, 8, 12, 30, 45, 60, 22, 18, 10, 7]  # spike around minute 5!
rolling = sliding_totals(per_minute, window_minutes=5)

print("🌊 Pure-Python Sliding Totals (5-minute window)")
print("=" * 55)
print(f"{'Minute':<8} {'This Minute':<14} {'Rolling 5m':<12}")
print("-" * 55)
for i, (m, r) in enumerate(zip(per_minute, rolling)):
    bar = "█" * (r // 10)
    print(f"{i:<8} {m:<14} {r:<12} {bar}")

print()
print("💡 The rolling total smooths out spikes and is what powers dashboards like:")
print("   'clicks in the last 5 minutes', 'CTR over the last hour', etc.")


## 🧮 Step 8 — Reconciliation & Lambda Architecture

Streaming is fast but **not always perfectly accurate**:

- A consumer might crash mid-batch and re-process some events (double-counting).
- Late-arriving events might be dropped if they miss the watermark.
- A bug in the aggregation code could silently drift over time.

The fix is the **Lambda Architecture**: run a second, slower, highly-accurate batch pipeline alongside the streaming one. Periodically **reconcile** the two. Whenever they disagree, the batch result wins and corrects the streaming aggregates.

```
Raw events (source of truth)
     │
     ├──▶  🚀 Streaming path   ──▶  click_aggregates  (fast, approximate)
     │                                    ▲
     │                                    │ overwrite on mismatch
     └──▶  🧮 Batch job (hourly) ─────────┘    (slow, exact)
```

Let's simulate a streaming drift — a few aggregate rows end up with the wrong count — and then run a reconciliation job that corrects them from the raw `click_events` table.


In [ ]:
# First, produce a fresh batch where raw events AND aggregates both exist.
reconcile_topic = f"ad-clicks-reconcile-{uuid.uuid4().hex[:8]}"  # unique per run for clean demos
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
if reconcile_topic not in admin.list_topics(timeout=10).topics:
    for t, f in admin.create_topics(
        [NewTopic(reconcile_topic, num_partitions=3, replication_factor=1)]
    ).items():
        f.result()

producer = Producer({"bootstrap.servers": KAFKA_BROKER})
base = datetime(2026, 3, 31, 11, 0, 0, tzinfo=timezone.utc)

# Clear and write a known set of raw events + aggregates
conn = get_db(); cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM click_events")
conn.commit()

raw_events = []
for minute in range(3):
    for ad_id in (1, 2):
        count = random.randint(5, 15)
        for _ in range(count):
            event_time = base + timedelta(minutes=minute, seconds=random.randint(0, 59))
            evt = {
                "ad_id": ad_id,
                "impression_id": str(uuid.uuid4()),
                "user_id": f"user_{random.randint(1,50)}",
                "ip_address": "10.0.0.1",
                "user_agent": "Mozilla/5.0",
                "event_time": event_time.isoformat(),
            }
            raw_events.append(evt)
            # Write the raw event to Postgres (source of truth)
            cur.execute(
                "INSERT INTO click_events (ad_id, impression_id, user_id, "
                "ip_address, user_agent, event_time) VALUES (%s,%s,%s,%s,%s,%s)",
                (evt["ad_id"], evt["impression_id"], evt["user_id"],
                 evt["ip_address"], evt["user_agent"], evt["event_time"]),
            )
            # And publish to Kafka for the streaming path
            producer.produce(reconcile_topic, key=str(ad_id), value=json.dumps(evt))

conn.commit(); conn.close()
producer.flush(timeout=10)

# Run the streaming aggregation (reuses aggregate_clicks + flush_to_postgres from Step 2/3)
_consumed, windows = aggregate_clicks(reconcile_topic, max_events=len(raw_events))
flush_to_postgres(windows)
print(f"✅ Produced {len(raw_events)} raw events and ran streaming aggregation.")


In [ ]:
# Now simulate a streaming BUG: one aggregate row ends up with the wrong count
# (e.g., the consumer crashed mid-flush and lost some events).

conn = get_db(); cur = conn.cursor()
cur.execute("""
    UPDATE click_aggregates
    SET click_count = click_count - 3
    WHERE window_start = '2026-03-31 11:00:00+00' AND ad_id = 1
""")
drifted_rows = cur.rowcount
conn.commit()
print(f"💥 Injected drift: reduced ad_id=1 @ 11:00 by 3 clicks ({drifted_rows} row)")

# Run the RECONCILIATION BATCH JOB.
# It recomputes aggregates from the raw click_events table (the source of truth)
# and overwrites any rows that disagree.
cur.execute("""
    WITH truth AS (
        SELECT
            ad_id,
            date_trunc('minute', event_time) AT TIME ZONE 'UTC' AS window_start,
            COUNT(*)                 AS true_count,
            COUNT(DISTINCT user_id)  AS true_unique
        FROM click_events
        GROUP BY ad_id, date_trunc('minute', event_time)
    )
    SELECT t.ad_id, t.window_start, t.true_count,
           COALESCE(ca.click_count, -1) AS streaming_count
    FROM truth t
    LEFT JOIN click_aggregates ca
      ON ca.ad_id = t.ad_id AND ca.window_start = t.window_start
    WHERE t.true_count <> COALESCE(ca.click_count, -1)
""")
mismatches = cur.fetchall()
print(f"🔍 Reconciliation found {len(mismatches)} mismatch(es):")
for ad_id, ws, truth, streaming in mismatches:
    delta = truth - streaming
    print(f"   ad={ad_id} window={ws} truth={truth} streaming={streaming} (Δ={delta:+d})")

# Repair the streaming aggregates from raw events (batch wins).
cur.execute("""
    INSERT INTO click_aggregates (ad_id, window_start, click_count, unique_users)
    SELECT
        ad_id,
        date_trunc('minute', event_time) AT TIME ZONE 'UTC',
        COUNT(*),
        COUNT(DISTINCT user_id)
    FROM click_events
    GROUP BY ad_id, date_trunc('minute', event_time)
    ON CONFLICT (ad_id, window_start) DO UPDATE
    SET click_count  = EXCLUDED.click_count,
        unique_users = EXCLUDED.unique_users,
        updated_at   = CURRENT_TIMESTAMP
""")
repaired = cur.rowcount
conn.commit(); conn.close()

print(f"🛠️  Repaired/refreshed {repaired} aggregate rows from raw events.")
print()
print("💡 In production this job runs hourly or daily. Advertisers see fast")
print("   streaming numbers immediately; a few hours later those numbers become")
print("   exact via reconciliation. That's the Lambda Architecture in action.")


## 🧹 Cleanup

In [ ]:
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM click_events")
conn.commit()
conn.close()
print("🧹 Cleaned up PostgreSQL tables")
print("   (Kafka topics still exist — that's fine for Notebook 3)")

## 📚 Summary

### Key Takeaways

1. **Pre-aggregation** trades storage for query speed — one row per ad per minute instead of millions of raw events
2. **Event time > processing time** — always bucket clicks by when they *happened*, not when you *received* them
3. **Tumbling windows** are fixed, non-overlapping time buckets (1 minute each in our case)
4. **Late events** are handled via UPSERT — the aggregate is updated even after the window has been flushed
5. **Flink** does all of this automatically in production with watermarks, state management, and exactly-once processing

### What's Next

In **Notebook 3**, we tackle **deduplication and fraud prevention**: how to stop the same click from being counted twice, how to sign impression IDs with HMAC, and how to use Redis as a fast dedup cache.